# 改善10候補 一括実験・提出ファイル作成

**目的**: 提出ファイルの作成に一直線。各候補で 1 本ずつ CSV を `outputs/submissions/` に保存する（分析用出力や CV だけの実験は行わない）。

**ベース**: 0.75493 を出したパイプライン（`get_baseline_data` + movie_title_info embedding + PCA 16 + doc_x_critic_te）。特徴数 55。38 特徴のみにしたい場合は `get_setup(..., use_best_pipeline=False)`。

**実装**: `lib/improvement_candidates.py` にロジックを集約。ノートは呼び出しのみ。

| # | 候補 | 提出ファイル | パート |
|---|------|-------------|--------|
| 1 | 疑似ラベル | submission_improvement_01_pseudo_label.csv | Part2 |
| 2 | 類似度TE (Jaccard top-k) | submission_improvement_02_similarity_te.csv | Part3 |
| 3 | LGB+XGB+CatBoost→Ridge スタッキング | submission_improvement_03_stacking.csv | Part2 |
| 4 | 既存提出のブレンド | submission_improvement_04_blend.csv | Part3 |
| 5 | scale_pos_weight | submission_improvement_05_scale_pos_weight.csv | Part1 |
| 6 | GroupKFold 予測 | submission_improvement_06_groupkfold.csv | Part1 |
| 7 | 特徴選択（重要度0除去） | submission_improvement_07_feature_selection.csv | Part1 |
| 8 | TF-IDF+SVD | submission_improvement_08_tfidf_svd.csv | Part2 |
| 9 | グラフ埋め込み | （スキップ） | Part3 |
| 10 | 弱点1列 (year_norm_x_critic_te) | submission_improvement_10_extra_col.csv | Part1 |

**実行**: Part1 → Part2 → Part3 の順で実行可。

## 共通セットアップ

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from lib.improvement_candidates import (
    get_setup,
    run_05_scale_pos_weight,
    run_10_extra_col,
    run_07_feature_selection,
    run_06_groupkfold,
    run_01_pseudo_label,
    run_03_stacking,
    run_03_stacking_batch,
    run_08_tfidf_svd,
    run_04_blend,
    run_02_similarity_te,
    list_improvement_submissions,
)

ctx = get_setup(seed=42, output_dir="outputs")

def _report(name, r):
    fname = Path(r["path"]).name if r.get("path") else "—"
    status = "OK" if r.get("ok") else r.get("message", "")
    print(f"  [{name}] → {fname}  ({status})")

print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")
print(f"時系列CV: {len(ctx.time_splits)} folds")
print(f"提出先: {ctx.submissions_dir}")

Train: 653,507, Test: 40,716, Features: 55
時系列CV: 4 folds
提出先: outputs/submissions


---
## Part1: 軽量（05, 10, 07, 06）

In [2]:
r05 = run_05_scale_pos_weight(ctx)
_report("05", r05)

r10 = run_10_extra_col(ctx)
_report("10", r10)

r07 = run_07_feature_selection(ctx)
print(f"  除去: {r07.get('n_dropped', 0)} 列 → 残り {r07.get('n_kept', 0)} 列")
_report("07", r07)

r06 = run_06_groupkfold(ctx)
print(f"  GroupKFold AUC: {r06.get('gkf_auc_mean', 0):.4f}")
_report("06", r06)

  [05] → submission_improvement_05_scale_pos_weight.csv  (OK)
  [10] → submission_improvement_10_extra_col.csv  (OK)
  除去: 0 列 → 残り 55 列
  [07] → submission_improvement_07_feature_selection.csv  (OK)
  GroupKFold AUC: 0.7388
  [06] → submission_improvement_06_groupkfold.csv  (OK)


---
## Part2: 中程度（01, 03, 08）

---
## 03 スタッキング 5 本（seed 42〜46）

同じ 55 特徴で LGB+XGB+CatBoost→Ridge を **seed を変えて 5 回**実行し、提出用 CSV を 5 本保存する。
- `submission_improvement_03_stacking_seed42.csv` … `submission_improvement_03_stacking_seed46.csv`
- xgboost / catboost が未導入の場合は先に `pip install xgboost catboost` を実行。

In [ ]:
# 5 本とも時間がかかります（各 seed で LGB+XGB+CatBoost を時系列 4-fold）
results_03_batch = run_03_stacking_batch(ctx)
print(f"  → {sum(1 for r in results_03_batch if r.get('ok'))} / {len(results_03_batch)} 本 OK")

In [3]:
r01 = run_01_pseudo_label(ctx)
if r01.get("n_pseudo", 0) == 0:
    print("  [01] 疑似ラベル: 0.95/0.05 を満たす行が0件のためベース予測を保存")
else:
    print(f"  [01] 疑似ラベル: {r01['n_pseudo']:,} 件を追加して再学習")
_report("01", r01)

r03 = run_03_stacking(ctx)
if r03.get("path"):
    _report("03", r03)
else:
    print("  [03] スキップ:", r03.get("message", "") + " → pip install xgboost catboost")

r08 = run_08_tfidf_svd(ctx)
_report("08", r08)

  [01] 疑似ラベル: 1,500 件を追加して再学習
  [01] → submission_improvement_01_pseudo_label.csv  (OK)
  [03] スキップ: xgboost/catboost 未導入: No module named 'xgboost' → pip install xgboost catboost
  [08] → submission_improvement_08_tfidf_svd.csv  (OK)


---
## Part3: 重い・オプション（04, 02, 09）

- **04**: `submission.csv`（train_baseline）と pca8/16/32（quick_embedding）があるとブレンド可能。
- **02**: 類似度TEは計算が重い。
- **09**: グラフ埋め込みは別実装。

In [4]:
r04 = run_04_blend(ctx)
if not r04.get("ok"):
    print(f"  [04] スキップ: {r04.get('message', '')}")
else:
    print(f"  [04] blend ({r04.get('n_blended', 0)}本)")
    _report("04", r04)

r02 = run_02_similarity_te(ctx, k=5)
_report("02", r02)

print("  [09] グラフ埋め込みはスキップ（ProNE/Node2Vec は別実装）")

  [04] スキップ: ブレンド用CSVが2本未満: ['submission_embedding_movie_title_info_pca16_doc_x_critic_te.csv', 'submission_embedding_movie_title_info_pca8.csv', 'submission_embedding_movie_title_info_pca16.csv', 'submission_embedding_movie_title_info_pca32.csv', 'submission.csv']


KeyboardInterrupt: 

---
## 提出ファイル一覧

In [ ]:
for p in list_improvement_submissions(ctx.submissions_dir):
    print(f"  {p.name}")

NameError: name 'list_improvement_submissions' is not defined